# Preparing your data for fine-tuning

Great, you've decided you to give fine-tuning a shot - you're going to need some data.  

## What type of data do we need? 

When you train a model, you are providing it with an input and the ideal corresponding output.  

In our fine-tuning use case, our inputs are essentially the user's prompts (eg. "a bit quieter please")

And the outputs are the corresponding function descriptions which should be returned from the model (eg. `{"function_name": "set_volume", "parameters": {"volume_level": 12}}`)


## Where should we get our data from? 

Some options: 

1. You could manually gather the dataset (actual requests from end users) and choose the desired output. 
2. Or you could create a synthetic dataset. 
3. Alternatively, there are open-source general-purpose datasets created for function-calling in general. https://huggingface.co/datasets/Salesforce/xlam-function-calling-60k/tree/main Note that you need to log in to see it. 


## Here's some we baked earlier ... 

We have prepared a dataset for this task, and stored it in a CSV file in the repo's '/Data' directory. 

Here's a snapshot: 

![](images/data_snapshot.png)

Can you see Excel doing its magic with the `explicitness` column? 

Open the file [/Data/train_data.csv](../Data/train_data.csv) and inspect the data - it's just a standard CSV. 


## Where did the dataset come from?

The dataset has been synthetically created using the script in [/data_generation/data_generation.ipynb](../data_generation/data_generation.ipynb). The script takes the function specification from [/Data/function_spec.json](../Data/function_spec.json), which has been manually created to represent a car control function within the radio domain. The script utilizes an LLM call (OpenAI) to generate multiple instances of user prompts and their function calls, varying emotional states and levels of explicitness.

## Inspecting the data with the Datasets library 

Let's take a look at a single sample from the training dataset to see what the structure is like. You can see that the columns in the CSV file (id, prompt, name, etc.) are now "attributes", or "keys" in the data sample.


In [ ]:
from datasets import load_dataset

# Load the training dataset from a CSV file.
# The "split='all'" option indicates that the entire dataset will be loaded without splitting it into train/test/validation.
train_dataset = load_dataset("csv", data_files="../Data/train_data.csv", split="all")

# Load the test dataset from a separate CSV file.
# This dataset will be used to evaluate the model's performance.
test_dataset = load_dataset("csv", data_files="../Data/test_data.csv", split="all")

# Check the form of the training dataset
train_dataset

<span style="color: green;">**QUESTION**</span>: Why do we need two dataset, one called train, and one called test?*

<details>
    <summary>Answer: Click to show</summary>

We need a separate test dataset to evaluate the performance of the model after training. The test dataset is not used during the training process, allowing us to assess how well the model generalizes to unseen data. By comparing the model's predictions on the test dataset with the actual values, we can determine its accuracy and effectiveness in real-world scenarios.
</details>

## Let's get more familiar with the Datasets samples 

Let's look at some of the other things you can do with the Datasets library. 

### Accessing individual rows using indexing

Dataset objects support direct indexing, similar to a list

In [ ]:
# Fetch the first sample directly using indexing
first_sample = train_dataset[0]

first_sample

### Extracting a few samples using .take()

The `.take(n)` method returns an iterable containing n samples from the dataset. This is useful for quick inspection without printing the entire dataset.

In [ ]:
# The `take(1)` method retrieves the first record.
data_sample = train_dataset.take(1)

# Print the sample to check its content and verify that the dataset is loaded correctly.
data_sample

<span style="color: green;">**QUESTION**</span>: 🤔 Can you guess what the following will output ? 

In [ ]:
# Print the prompt to check its content and verify that the dataset is loaded correctly.
train_dataset.take(4)["prompt"]

In [ ]:
train_dataset[2]

### Removing unused columns

Sometimes, datasets contain columns that aren’t needed for training. You can remove them using `.remove_columns()`.

In [ ]:
train_dataset = train_dataset.remove_columns(['explicitness','emotional_state'])

### Converting to Pandas 🐼

You can easily convert a Dataset instance to a pandas, where complex transformations are simpler.  

In [ ]:
df = train_dataset.to_pandas()

df.head()


### Using .map() for efficient transformations

The `.map()` method applies a function directly to each member of the dataset without needing to convert it to Pandas. This is more memory-efficient for large datasets.

In [ ]:
shout_dataset = train_dataset.map(lambda x: {"prompt": x["prompt"].strip().upper()})

shout_dataset[0]

## Summary 

You should hopefully now be familiar with the type of data you need for finetuning.  

And you've also become BFFs with the Datasets library along the way. 💕

----

## (Optional) Validating and cleaning your dataset 

You should always validate and if necessary clean your dataset.  Here, we discover that there are some invalid entries. 

In [ ]:
import json

import pandas as pd


def identify_invalid_json(dataframe: pd.DataFrame) -> list[tuple[int, str, Exception]]:
    # Initialize an empty list to store any errors encountered during JSON validation
    errors = []

    # Iterate through the "ground_truth" column to check if each entry is valid JSON
    for row in dataframe.itertuples():
        try:
            # Attempt to parse the JSON string
            json.loads(row.ground_truth)
        except (json.JSONDecodeError, TypeError) as e:
            # If there's an error (e.g., invalid JSON or wrong type), add it to the errors list
            # Store the row index, the invalid entry, and the exception
            errors.append(
                {
                    "id": row.id,
                    "ground_truth": row.ground_truth,
                    "exception": e,
                }
            )

    return errors

# Load the training data as pandas Dataframes
raw_train_data_csv = pd.read_csv("../Data/train_data.csv")
raw_test_data_csv = pd.read_csv("../Data/test_data.csv")

# Identify the invalid values
invalid_json_train = identify_invalid_json(raw_train_data_csv)
invalid_json_test = identify_invalid_json(raw_test_data_csv)

# Print the errors to inspect the issues in the trainning dataset
for entry in invalid_json_train:
    print(
        "\n".join(
            [
                f"Row: {entry['id']}",
                f"Invalid entry: {entry['ground_truth']}",
                f"Error: {entry['exception']}",
                "===",
            ]
        )
    )


There are 3 entries in the training dataset that are not in the correct format.
- The first 2 are invalid jsons
- The last one is just empty

Luckily, there are no invalid entries in the test dataset. But this only tells us if the ground truth is in the correct format - we did not check if the content of the function call is correct!

We will now remove the invalid entries from the dataset and save the cleaned version:

In [ ]:
invalid_train_indices = [entry["id"] for entry in invalid_json_train]
invalid_test_indices = [entry["id"] for entry in invalid_json_test]

# Remove invalid entries from the datasets using pandas filtering
clean_train_data = raw_train_data_csv.loc[~raw_train_data_csv["id"].isin(invalid_train_indices)]
clean_test_data = raw_test_data_csv.loc[~raw_test_data_csv["id"].isin(invalid_test_indices)]

# Save the cleaned data back to CSV files
clean_train_data.to_csv("../Data/clean_train_data.csv", index=False)
clean_test_data.to_csv("../Data/clean_test_data.csv", index=False)

print(f"Removed {len(invalid_train_indices)} invalid entries from the training dataset. Remaining: {len(train_dataset)}")
print(f"Removed {len(invalid_test_indices)} invalid entries from the test dataset. Remaining: {len(test_dataset)}")

print(f"Saved cleaned training dataset with {len(clean_train_data)} rows.")
print(f"Saved cleaned test dataset with {len(clean_test_data)} rows.")

----

<span style="color: green;">**QUESTION**</span>: How much data do you need for fine-tuning?

<details>
    <summary>Click to show the answer</summary>

As always, it depends. Factors that can determine the size of your dataset would be:
- The complexity of the task
- The size of the model
- The quality of the finetuning data
- The similarity of the data to the task you are fine-tuning for
- How accurate do you need the model to be
- Your finetuning technique

A good rule of thumb, finger in the air guesstimate is start with around 100 examples per **scenario** your model needs to capture. Then, depending on your analysis of the model performance, you can decide if you need more data or not. 
</details>